# Imports

In [0]:
import pickle
import subprocess
from datetime import datetime
from pathlib import Path
from mlflow import MlflowClient
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import unicodedata
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import unicodedata
import re

# Parâmetros do notebook

In [0]:
# dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")

In [0]:
dbutils.widgets.text("id_projeto", "endometriose", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

print(f"environment_tbl: {environment_tbl}")

In [0]:
tbl_entrada = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_entrada"
tbl_pre_processamento = f"{environment_tbl}tbl_gold_modelo_{id_projeto}_pre_processamento"

In [0]:
# output = subprocess.check_output("pwd", shell=True)
# current_folder = output.decode().strip()

In [0]:
def print_params():
    print(f"Data Execução Modelo : {data_execucao_modelo}")
    print(f"id_projeto           : {id_projeto}")
    # print(f"current_folder       : {current_folder}")
    print(f"environment          : {environment}")
    print(f"environment_tbl      : {environment_tbl}")
    print(f"tbl_entrada          : {tbl_entrada}")
    print(f"tbl_pre_processamento: {tbl_pre_processamento}")

print_params()

##Funções

In [0]:
# # Definindo a função clean_laudo sem o uso de striprtf
# def clean_laudo(str):
#     if str is None:
#         return ""

#     def replace_accents_characters(str):
#         if str is None:
#             return ""
#         return unicodedata.normalize('NFKD', str).encode('ascii', 'ignore').decode('utf-8')

#     def preprocess_text(text):
#         return "\n".join([line for line in text.splitlines() if line.strip()])

#     def remove_specific_text(text):
#         return text.replace('este laudo pode nao estar completo aqui em funcao do padrao de arquivo "rtf" utilizado. recomendamos a sua visualizacao no sistema da radiologia "webris".', '')

#     str_mod = preprocess_text(str)
#     str_mod = str_mod.replace("\n", " xxxenterxxx ")
#     str_split = str_mod.split()
#     str_mod = " ".join(str_split)
#     str_mod = str_mod.replace(" xxxenterxxx ", "\n")
#     str_mod = replace_accents_characters(str_mod)
#     str_mod = str_mod.lower()
#     str_mod = remove_specific_text(str_mod)
#     str_mod = str_mod.replace("endometriomas", "endometriose")
#     str_mod = str_mod.replace("endometrioma", "endometriose")
    
#     return str_mod

# # Registrando a função clean_laudo como UDF no Spark
# clean_laudo_udf = udf(clean_laudo, StringType())
# spark.udf.register("clean_laudo_udf", clean_laudo_udf)

In [0]:
def replace_accents_characters(str):
    if str is None:
        return ""
    return unicodedata.normalize('NFKD', str).encode('ascii', 'ignore').decode('utf-8')

def preprocess_text(text):
    return "\n".join([line for line in text.splitlines() if line.strip()])

def remove_specific_text(text):
    return text.replace('este laudo pode nao estar completo aqui em funcao do padrao de arquivo "rtf" utilizado. recomendamos a sua visualizacao no sistema da radiologia "webris".', '')

def replaces_caracteres(laudo_tratado):
    ltratado = laudo_tratado.replace('/',' / ')
    ltratado = ltratado.replace('?',' ')
    return ltratado

def replaces_lt(laudo_tratado):
    lt = laudo_tratado.replace('endometrioma','endometriose')
    lt = lt.replace('endometriomas','endometriose')
    lt = lt.replace('endometrioses','endometriose')
    lt = lt.replace('endometrioticos','endometriose')
    lt = lt.replace('endometriotico','endometriose')
    return lt


# Definindo a função clean_laudo sem o uso de striprtf
def clean_laudo(str):
    if str is None:
        return ""

    str_mod = preprocess_text(str)
    str_mod = str_mod.replace("\n", " xxxenterxxx ")
    str_split = str_mod.split()
    str_mod = " ".join(str_split)
    str_mod = str_mod.replace(" xxxenterxxx ", "\n")
    str_mod = replace_accents_characters(str_mod)
    str_mod = str_mod.lower()
    str_mod = remove_specific_text(str_mod)
    str_mod = replaces_lt(str_mod)
    str_mod = replaces_caracteres(str_mod)
    
    return str_mod

# Registrando a função clean_laudo como UDF no Spark
clean_laudo_udf = udf(clean_laudo, StringType())
spark.udf.register("clean_laudo_udf", clean_laudo_udf)

In [0]:
lista_sintomas = [
    "dor pelvica cronica",
    "dismenorreia",
    "dispareunia",
    "pesquisa de infertilidade",
    "infertilidade",
    "urgencia miccional",
    "disuria",
    "hematuria",
]

pattern_sintomas = '|'.join([re.escape(term) for term in lista_sintomas])
pattern_sintomas = f"({pattern_sintomas})"

lista_achados = [
    'endometrioma',
    'dilatacao unilateral do ureter',
    'dilatacao unilateral ureter',
    'espessamento do ligamento uterossacro',
    'ligamento uterossacro',
    'ligamentos uterossacro',
    'ligamento uterossacros',
    'ligamentos uterossacros',
    'espessamento dos ligamentos uterossacros',
    'espessamento do ligamento uterossacro',
    'espessamento do ligamento uterossacros',
    'espessamentos dos ligamentos uterossacros',
    'espessamento signifivcativo dos ligamentos uterossacros',
    'espessamento irregular do ligamento uterossacro',
    'espessamento na topografia dos ligamentos uterossacros',
    'espessando os ligamentos uterossacros',
    'deflexinal do ligamento uterossacro',
    'espessamento, hipossinal e irregularidade da porcao intraperitoneal dos ligamentos uterossacros',
    'espessamento ligamento uterossacro',
    'espessamento ligamentos uterossacros',
    'espessamento ligamento uterossacros',
    'endometriotica',
    'endometriotico',
    'endometriose',
    'lesões peritoneais',
    'lesões tubárias',
    'lesões ovarianas',
    'lesões infiltrativas'
]

pattern_achados = '|'.join([re.escape(term) for term in lista_achados])
pattern_achados = f"({pattern_achados})"

# Data

In [0]:
query = f"""
    SELECT
        dataExecucaoModelo,
        idPredicao,
        idExame,
        idPaciente,
        descricaoProcedimento,
        laudoOriginal,
        date(dataExame) as dataExame
    from ia.{tbl_entrada}
    where dataExecucaoModelo = '{data_execucao_modelo}'
      and laudoDuplicado = 1
"""

df_spark = spark.sql(query)

if df_spark.rdd.isEmpty():
  dbutils.notebook.exit("Não há dados para serem processados! Execução cancelada!")

# Registrar o dataframe como uma visão temporária
df_spark.createOrReplaceTempView("temp_df_spark")

# Criando a visão temporária cte_filtro para normalizar os textos
query_cte_filtro = """
CREATE OR REPLACE TEMP VIEW cte_filtro AS
select
    *,
    lower(translate(
        laudoOriginal,
        'ÀÁÂÃÄÅàáâãäåÇçÈÉÊËèéêëÌÍÎÏìíîïÒÓÔÕÖòóôõöÙÚÛÜùúûüÝý',
        'AAAAAAaaaaaaCcEEEEeeeeIIIIiiiiOOOOOoooooUUUUuuuuYy'
    )) as laudoOriginalLimpo
  from temp_df_spark
"""

# Executa a query para criar a visão temporária cte_filtro
spark.sql(query_cte_filtro)

In [0]:
query_cte_laudos = """
CREATE OR REPLACE TEMP VIEW cte_laudos AS
select
    *,
    clean_laudo_udf(laudoOriginalLimpo) as laudoTratado
from cte_filtro
"""

spark.sql(query_cte_laudos)
df_cte_laudos = spark.sql("SELECT * FROM cte_laudos")

In [0]:
query_cte_achados = f"""
CREATE OR REPLACE TEMP VIEW cte_achados AS
select
    *,
    regexp_extract_all(laudoTratado, '{pattern_sintomas}') AS listaSintomas,
    regexp_extract_all(laudoTratado, '{pattern_achados}') AS listaAchados
from cte_laudos
"""

spark.sql(query_cte_achados)
df_cte_achados = spark.sql("SELECT * FROM cte_achados")

In [0]:
query_final = """
select
  *,
  case
    when array_size(listaSintomas) = 0 and array_size(listaAchados) = 0
    then '1 - Sem achado e Sem Sintomas'
    when array_size(listaSintomas) > 0 and array_size(listaAchados) = 0
    then '2 - Só Sintoma'
    when array_size(listaSintomas) = 0 and array_size(listaAchados) > 0
    then '3 - Só Achado'
    when array_size(listaSintomas) > 0 and array_size(listaAchados) > 0
    then '4 - Achado e Sintoma'
    else 'Erro'
  end as classificacao
from cte_achados
"""

df_final = spark.sql(query_final)
#df_final_pandas = df_final.toPandas()

# Exibindo o dataframe final em Pandas
#print(df_final_pandas)

In [0]:
# contando linhas e colunas
num_linhas = df_final.count()
num_colunas = len(df_final.columns)

print("O dataframe tem {} linhas e {} colunas".format(num_linhas, num_colunas))

In [0]:
from pyspark.sql.functions import col, size

count_non_empty = df_final.filter((size(col("listaSintomas")) > 0) | (size(col("listaAchados")) > 0)).count()

print(count_non_empty)

In [0]:
# df_final.createOrReplaceTempView("df_final_view")

In [0]:
# spark.sql("""
#     SELECT classificacao, COUNT(1) AS qtd
#     FROM df_final_view
#     GROUP BY classificacao
# """).show()

In [0]:
# df_filtered = df_final.filter(df_final.classificacao != "1 - Sem achado e Sem Sintomas")

In [0]:
df_filtered = df_final#.filter(df_final.classificacao != "1 - Sem achado e Sem Sintomas")

In [0]:
# # contando linhas e colunas
# num_linhas = df_filtered.count()
# num_colunas = len(df_filtered.columns)

# print("O dataframe tem {} linhas e {} colunas".format(num_linhas, num_colunas))

In [0]:
df_filtered.display()

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [0]:
def extrair_motivos(texto):
    indice_motivo = texto.find('tecnica:')
    if indice_motivo != -1:
        return texto[indice_motivo:]
    return texto


# def extrair_motivos(texto):
#     indice_motivo = texto.find('tecnica:')

#     if indice_motivo != -1:
#         texto = texto[indice_motivo:]

#         return texto.strip()
#     else:
#         return texto

# extrair_motivos_udf = udf(extrair_motivos, StringType())
# spark.udf.register("extrair_motivos_udf", extrair_motivos_udf)

In [0]:
extrair_motivos_udf = udf(extrair_motivos, StringType())

In [0]:
df_filtered = df_filtered.withColumn('laudoResumido', extrair_motivos_udf(df_filtered['laudoTratado']))

In [0]:
df_filtered.display()

In [0]:
df_pandas = df_filtered.toPandas()
df_pandas.shape

In [0]:
df_pandas['classificacao'].value_counts()

In [0]:
df_pandas.head(1)

In [0]:
df_filtered.schema

# Salva os dados na tabela de pré-processamento

In [0]:
# schema = StructType(
#     [
#         StructField("dataExecucaoModelo", DateType(), True),
#         StructField("idPredicao", StringType(), True),
#         StructField("idExame", StringType(), True),
#         StructField("tokens", ArrayType(StringType(), True), True),
#         StructField("resultadosBigrams", ArrayType(StringType(), True), True),
#         StructField("resultadosTrigrams", ArrayType(StringType(), True), True),
#         StructField("textoTrigrams", StringType(), True),
#         StructField("score", DoubleType(), True),
#         StructField("decil", StringType(), True),

#         StructField("dataCargaPreProcessamento", TimestampType(), True),
#         StructField("nomeModeloScore", StringType(), True),
#         StructField("estagioModeloScore", StringType(),True),
#         StructField("versaoModeloScore", StringType(), True),
#         StructField("idTreinamentoScore", StringType(), True),
#     ]
# )

# spark_df = spark.createDataFrame(df_pandas[schema.fieldNames()], schema)
# spark_df.display()

In [0]:
tbl_pre_processamento

In [0]:
table_exists = spark.catalog.tableExists(f"ia.{tbl_pre_processamento}")

if table_exists:
    spark.sql(f"""
        delete from ia.{tbl_pre_processamento}
        where dataExecucaoModelo = '{data_execucao_modelo}'
    """).display()

In [0]:
df_filtered.write.mode("append").option("mergeSchema", "true").partitionBy("dataExecucaoModelo").saveAsTable(f"ia.{tbl_pre_processamento}")

In [0]:
spark.sql(f"vacuum ia.{tbl_pre_processamento}").display()
spark.sql(f"optimize ia.{tbl_pre_processamento}").display()
spark.sql(f"analyze table ia.{tbl_pre_processamento} compute statistics").display()

In [0]:
# criar a tabela = setup
# puxar os dados = entrada
# pre processamento = pre processamento
# criar a saida = saida

In [0]:
spark.sql(f"""
    select
        dataExecucaoModelo,
        count(1) as qtd
    from ia.{tbl_pre_processamento}
    group by all
    order by dataExecucaoModelo desc
""").display()

In [0]:
spark.sql(f"""
    select
        idExame,
        count(1) as qtd
    from ia.{tbl_pre_processamento}
    group by all
    having qtd > 1
""").display()

In [0]:
dbutils.notebook.exit("Fim do processamento!")

# Fim do processamento!